In [5]:
import pandas as pd

# =========================
# 1. LOAD RAW CSV FILES
# =========================
happy = pd.read_csv("/Users/ale/Desktop/MySQL/Project-3/world_happiness_report.csv")
econ = pd.read_csv("/Users/ale/Desktop/MySQL/Project-3/Economic Indicators And Inflation.csv")

# Clean column names
happy.columns = happy.columns.str.strip()
econ.columns = econ.columns.str.strip()

# =========================
# 2. STANDARDIZE COUNTRY NAMES
# =========================
country_map = {
    "USA": "United States",
    "US": "United States",
    "UK": "United Kingdom"
}

happy["Country"] = happy["Country"].replace(country_map).str.strip()
econ["Country"] = econ["Country"].replace(country_map).str.strip()

# =========================
# 3. FIX YEAR DATA TYPE
# =========================
happy["Year"] = pd.to_numeric(happy["Year"], errors="coerce")
econ["Year"] = pd.to_numeric(econ["Year"], errors="coerce")

happy = happy.dropna(subset=["Year", "Country"])
econ = econ.dropna(subset=["Year", "Country"])

happy["Year"] = happy["Year"].astype(int)
econ["Year"] = econ["Year"].astype(int)

# =========================
# 4. REMOVE DUPLICATES
# =========================
happy = happy.drop_duplicates()
econ = econ.drop_duplicates()

# =========================
# 5. CONVERT NUMERIC COLUMNS
# =========================
for col in happy.columns:
    if col not in ["Country", "Year"]:
        happy[col] = pd.to_numeric(happy[col], errors="coerce")

for col in econ.columns:
    if col not in ["Country", "Year"]:
        econ[col] = pd.to_numeric(econ[col], errors="coerce")

# =========================
# 6. CREATE DIMENSION TABLES
# =========================
countries = pd.DataFrame({
    "country_name": sorted(set(happy["Country"]).union(set(econ["Country"])))
})
countries.insert(0, "country_id", range(1, len(countries) + 1))

years = pd.DataFrame({
    "year_id": sorted(set(happy["Year"]).union(set(econ["Year"])))
})

# =========================
# 7. MERGE IDs INTO RAW DATA
# =========================
happy = happy.merge(countries, left_on="Country", right_on="country_name", how="left")
happy = happy.merge(years, left_on="Year", right_on="year_id", how="left")

econ = econ.merge(countries, left_on="Country", right_on="country_name", how="left")
econ = econ.merge(years, left_on="Year", right_on="year_id", how="left")

# =========================
# 8. CREATE HAPPINESS TABLE
# =========================
happiness_metrics = happy[[
    "country_id",
    "year_id",
    "Happiness_Score",
    "Life_Satisfaction",
    "Social_Support",
    "Healthy_Life_Expectancy",
    "Freedom",
    "Generosity",
    "Corruption_Perception",
    "Public_Trust",
    "Mental_Health_Index",
    "Work_Life_Balance",
    "Climate_Index"
]].copy()

happiness_metrics.insert(0, "happiness_id", range(1, len(happiness_metrics) + 1))

# =========================
# 9. CREATE SOCIAL/ECONOMIC TABLE
# =========================
social_economic_metrics = happy[[
    "country_id",
    "year_id",
    "GDP_per_Capita",
    "Population",
    "Urbanization_Rate",
    "Education_Index",
    "Unemployment_Rate",
    "Employment_Rate",
    "Income_Inequality",
    "Public_Health_Expenditure",
    "Internet_Access",
    "Crime_Rate",
    "Political_Stability"
]].copy()

social_economic_metrics.insert(0, "social_id", range(1, len(social_economic_metrics) + 1))

# Rename columns to match SQL table nicely
social_economic_metrics.columns = [
    "social_id",
    "country_id",
    "year_id",
    "gdp_per_capita",
    "population",
    "urbanization_rate",
    "education_index",
    "unemployment_rate",
    "employment_rate",
    "income_inequality",
    "public_health_expenditure",
    "internet_access",
    "crime_rate",
    "political_stability"
]

# =========================
# 10. CREATE MACROECONOMIC TABLE
# =========================
macroeconomic_indicators = econ[[
    "country_id",
    "year_id",
    "GDP (in billion USD)",
    "Inflation Rate (%)",
    "Unemployment Rate (%)",
    "Economic Growth (%)"
]].copy()

macroeconomic_indicators.columns = [
    "country_id",
    "year_id",
    "gdp_billion_usd",
    "inflation_rate_pct",
    "unemployment_rate_pct",
    "economic_growth_pct"
]

macroeconomic_indicators.insert(0, "macro_id", range(1, len(macroeconomic_indicators) + 1))

# =========================
# 11. OPTIONAL: HANDLE NULLS
# =========================
# Fill numeric nulls with median
for df in [happiness_metrics, social_economic_metrics, macroeconomic_indicators]:
    numeric_cols = df.select_dtypes(include="number").columns
    for col in numeric_cols:
        if col not in ["happiness_id", "social_id", "macro_id", "country_id", "year_id"]:
            df[col] = df[col].fillna(df[col].median())

# =========================
# 12. EXPORT ALL CSV FILES
# =========================
output_folder = "/Users/ale/Desktop/MySQL/Project-3/"

countries.to_csv(output_folder + "countries.csv", index=False)
years.to_csv(output_folder + "years.csv", index=False)
happiness_metrics.to_csv(output_folder + "happiness_metrics.csv", index=False)
social_economic_metrics.to_csv(output_folder + "social_economic_metrics.csv", index=False)
macroeconomic_indicators.to_csv(output_folder + "macroeconomic_indicators.csv", index=False)

print("All 5 CSV files created successfully.")

All 5 CSV files created successfully.
